# Avaliação de Data Quality — voebem.silver.vra

Avaliação completa da qualidade dos dados e do aspecto de negócio da tabela **voebem.silver.vra**, que armazena as etapas de voo do Voo Regular Atual (VRA) da ANAC. A tabela é um espelho governado do bronze com tipagem, data e hora separadas, e as três métricas de atraso em aritmética pura.

## 1. Visão Geral e Completude

Contagem total de registros, range de datas e verificação de valores nulos/vazios por coluna.

In [0]:
%sql
-- Contagem total, range de datas e volume por ano-mes
SELECT
  COUNT(*) AS total_registros,
 MIN(partida_prevista_data) AS data_minima,
  MAX(partida_prevista_data) AS data_maxima,
  COUNT(DISTINCT _arquivo_origem) AS arquivos_distintos
FROM voebem.silver.vra;

In [0]:
%sql
-- Volume de registros por ano-mes (cobertura temporal)
SELECT
  DATE_FORMAT(partida_prevista_data, 'yyyy-MM') AS ano_mes,
  COUNT(*) AS qtd_registros,
  COUNT(DISTINCT _arquivo_origem) AS arquivos
FROM voebem.silver.vra
GROUP BY DATE_FORMAT(partida_prevista_data, 'yyyy-MM')
ORDER BY ano_mes;

In [0]:
%sql
-- Completude: contagem de nulos e vazios por coluna de negocio
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN icao_empresa IS NULL OR TRIM(icao_empresa) = '' THEN 1 ELSE 0 END) AS icao_empresa_nulo,
  SUM(CASE WHEN numero_voo IS NULL OR TRIM(numero_voo) = '' THEN 1 ELSE 0 END) AS numero_voo_nulo,
  SUM(CASE WHEN codigo_di IS NULL OR TRIM(codigo_di) = '' THEN 1 ELSE 0 END) AS codigo_di_nulo,
  SUM(CASE WHEN codigo_tipo_linha IS NULL OR TRIM(codigo_tipo_linha) = '' THEN 1 ELSE 0 END) AS codigo_tipo_linha_nulo,
  SUM(CASE WHEN icao_origem IS NULL OR TRIM(icao_origem) = '' THEN 1 ELSE 0 END) AS icao_origem_nulo,
  SUM(CASE WHEN icao_destino IS NULL OR TRIM(icao_destino) = '' THEN 1 ELSE 0 END) AS icao_destino_nulo,
  SUM(CASE WHEN partida_prevista IS NULL THEN 1 ELSE 0 END) AS partida_prevista_nulo,
  SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END) AS partida_real_nulo,
  SUM(CASE WHEN chegada_prevista IS NULL THEN 1 ELSE 0 END) AS chegada_prevista_nulo,
  SUM(CASE WHEN chegada_real IS NULL THEN 1 ELSE 0 END) AS chegada_real_nulo,
  SUM(CASE WHEN situacao_voo IS NULL OR TRIM(situacao_voo) = '' THEN 1 ELSE 0 END) AS situacao_voo_nulo,
  SUM(CASE WHEN codigo_justificativa IS NULL OR TRIM(codigo_justificativa) = '' THEN 1 ELSE 0 END) AS codigo_justificativa_nulo,
  SUM(CASE WHEN atraso_partida_min IS NULL THEN 1 ELSE 0 END) AS atraso_partida_min_nulo,
  SUM(CASE WHEN atraso_chegada_min IS NULL THEN 1 ELSE 0 END) AS atraso_chegada_min_nulo,
  SUM(CASE WHEN minutos_recuperados IS NULL THEN 1 ELSE 0 END) AS minutos_recuperados_nulo
FROM voebem.silver.vra;

In [0]:
%sql
-- Completude em percentual (colunas-chave)
SELECT
  ROUND(100.0 * SUM(CASE WHEN icao_empresa IS NULL OR TRIM(icao_empresa) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_icao_empresa_ausente,
  ROUND(100.0 * SUM(CASE WHEN numero_voo IS NULL OR TRIM(numero_voo) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_numero_voo_ausente,
  ROUND(100.0 * SUM(CASE WHEN codigo_di IS NULL OR TRIM(codigo_di) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_codigo_di_ausente,
  ROUND(100.0 * SUM(CASE WHEN codigo_tipo_linha IS NULL OR TRIM(codigo_tipo_linha) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_codigo_tipo_linha_ausente,
  ROUND(100.0 * SUM(CASE WHEN icao_origem IS NULL OR TRIM(icao_origem) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_icao_origem_ausente,
  ROUND(100.0 * SUM(CASE WHEN icao_destino IS NULL OR TRIM(icao_destino) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_icao_destino_ausente,
  ROUND(100.0 * SUM(CASE WHEN partida_prevista IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_partida_prevista_ausente,
  ROUND(100.0 * SUM(CASE WHEN situacao_voo IS NULL OR TRIM(situacao_voo) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_situacao_voo_ausente
FROM voebem.silver.vra;

## 2. Unicidade e Duplicatas

Verificação de registros duplicados pela chave natural: `icao_empresa + numero_voo + codigo_di + partida_prevista`.

In [0]:
%sql
-- Duplicatas pela chave natural (icao_empresa, numero_voo, codigo_di, partida_prevista)
SELECT
  COUNT(*) AS total_linhas,
  COUNT(*) - COUNT(DISTINCT icao_empresa, numero_voo, codigo_di, partida_prevista) AS linhas_duplicadas,
  COUNT(DISTINCT icao_empresa, numero_voo, codigo_di, partida_prevista) AS chaves_unicas
FROM voebem.silver.vra
WHERE icao_empresa IS NOT NULL AND numero_voo IS NOT NULL AND codigo_di IS NOT NULL AND partida_prevista IS NOT NULL;

In [0]:
%sql
-- Top 20 grupos duplicados pela chave natural
SELECT icao_empresa, numero_voo, codigo_di, partida_prevista, COUNT(*) AS qtd
FROM voebem.silver.vra
WHERE icao_empresa IS NOT NULL AND numero_voo IS NOT NULL AND codigo_di IS NOT NULL AND partida_prevista IS NOT NULL
GROUP BY icao_empresa, numero_voo, codigo_di, partida_prevista
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

## 3. Consistência e Validade de Formatos

Validação dos formatos esperados: `icao_empresa` (3 letras), `icao_origem/destino` (4 letras), `codigo_tipo_linha` (N/C/I/G), `situacao_voo` (REALIZADO/CANCELADO), e padronização de colunas categóricas.

In [0]:
%sql
-- Validacao de formatos ICAO (empresa=3 letras, origem/destino=4 letras)
SELECT
  SUM(CASE WHEN icao_empresa IS NOT NULL AND TRIM(icao_empresa) != '' AND NOT RLIKE(icao_empresa, '^[A-Z]{3}$') THEN 1 ELSE 0 END) AS icao_empresa_formato_invalido,
  SUM(CASE WHEN icao_empresa IS NOT NULL AND TRIM(icao_empresa) != '' AND RLIKE(icao_empresa, '^[A-Z]{3}$') THEN 1 ELSE 0 END) AS icao_empresa_formato_valido,
  SUM(CASE WHEN icao_origem IS NOT NULL AND TRIM(icao_origem) != '' AND NOT RLIKE(icao_origem, '^[A-Z]{4}$') THEN 1 ELSE 0 END) AS icao_origem_formato_invalido,
  SUM(CASE WHEN icao_origem IS NOT NULL AND TRIM(icao_origem) != '' AND RLIKE(icao_origem, '^[A-Z]{4}$') THEN 1 ELSE 0 END) AS icao_origem_formato_valido,
  SUM(CASE WHEN icao_destino IS NOT NULL AND TRIM(icao_destino) != '' AND NOT RLIKE(icao_destino, '^[A-Z]{4}$') THEN 1 ELSE 0 END) AS icao_destino_formato_invalido,
  SUM(CASE WHEN icao_destino IS NOT NULL AND TRIM(icao_destino) != '' AND RLIKE(icao_destino, '^[A-Z]{4}$') THEN 1 ELSE 0 END) AS icao_destino_formato_valido
FROM voebem.silver.vra;

In [0]:
%sql
-- Valores distintos de codigo_tipo_linha
SELECT codigo_tipo_linha, COUNT(*) AS qtd
FROM voebem.silver.vra
GROUP BY codigo_tipo_linha
ORDER BY qtd DESC;

In [0]:
%sql
-- Valores distintos de codigo_di
SELECT codigo_di, COUNT(*) AS qtd
FROM voebem.silver.vra
GROUP BY codigo_di
ORDER BY qtd DESC;

In [0]:
%sql
-- Valores distintos de situacao_voo
SELECT situacao_voo, COUNT(*) AS qtd,
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM voebem.silver.vra), 2) AS pct
FROM voebem.silver.vra
GROUP BY situacao_voo
ORDER BY qtd DESC;

In [0]:
%sql
-- Distribuicao de codigo_justificativa (esperado: vazio desde abr/2020)
SELECT
  CASE
    WHEN codigo_justificativa IS NULL OR TRIM(codigo_justificativa) = '' THEN 'vazio/null'
    ELSE codigo_justificativa
  END AS codigo_justificativa,
  COUNT(*) AS qtd
FROM voebem.silver.vra
GROUP BY CASE
    WHEN codigo_justificativa IS NULL OR TRIM(codigo_justificativa) = '' THEN 'vazio/null'
    ELSE codigo_justificativa
  END
ORDER BY qtd DESC
LIMIT 20;

## 4. Consistência entre Colunas Derivadas

Validação de que as colunas derivadas (`_data`, `_hora`) e as métricas de atraso foram calculadas corretamente a partir dos timestamps originais.

In [0]:
%sql
-- Verificar consistencia: _data e _hora extraidos dos timestamps
SELECT
  SUM(CASE WHEN partida_prevista IS NOT NULL AND partida_prevista_data != CAST(partida_prevista AS DATE) THEN 1 ELSE 0 END) AS partida_prevista_data_inconsistente,
  SUM(CASE WHEN partida_prevista IS NOT NULL AND partida_prevista_hora != DATE_FORMAT(partida_prevista, 'HH:mm') THEN 1 ELSE 0 END) AS partida_prevista_hora_inconsistente,
  SUM(CASE WHEN partida_real IS NOT NULL AND partida_real_data != CAST(partida_real AS DATE) THEN 1 ELSE 0 END) AS partida_real_data_inconsistente,
  SUM(CASE WHEN partida_real IS NOT NULL AND partida_real_hora != DATE_FORMAT(partida_real, 'HH:mm') THEN 1 ELSE 0 END) AS partida_real_hora_inconsistente,
  SUM(CASE WHEN chegada_prevista IS NOT NULL AND chegada_prevista_data != CAST(chegada_prevista AS DATE) THEN 1 ELSE 0 END) AS chegada_prevista_data_inconsistente,
  SUM(CASE WHEN chegada_prevista IS NOT NULL AND chegada_prevista_hora != DATE_FORMAT(chegada_prevista, 'HH:mm') THEN 1 ELSE 0 END) AS chegada_prevista_hora_inconsistente,
  SUM(CASE WHEN chegada_real IS NOT NULL AND chegada_real_data != CAST(chegada_real AS DATE) THEN 1 ELSE 0 END) AS chegada_real_data_inconsistente,
  SUM(CASE WHEN chegada_real IS NOT NULL AND chegada_real_hora != DATE_FORMAT(chegada_real, 'HH:mm') THEN 1 ELSE 0 END) AS chegada_real_hora_inconsistente
FROM voebem.silver.vra;

In [0]:
%sql
-- Verificar consistencia das metricas de atraso
-- atraso_partida_min = (partida_real - partida_prevista) em minutos
-- atraso_chegada_min = (chegada_real - chegada_prevista) em minutos
-- minutos_recuperados = atraso_partida_min - atraso_chegada_min
SELECT
  SUM(CASE WHEN partida_real IS NOT NULL AND partida_prevista IS NOT NULL
       AND atraso_partida_min IS NOT NULL
       AND atraso_partida_min != CAST(ROUND((CAST(partida_real AS LONG) - CAST(partida_prevista AS LONG)) / 60.0) AS INT)
       THEN 1 ELSE 0 END) AS atraso_partida_inconsistente,
  SUM(CASE WHEN chegada_real IS NOT NULL AND chegada_prevista IS NOT NULL
       AND atraso_chegada_min IS NOT NULL
       AND atraso_chegada_min != CAST(ROUND((CAST(chegada_real AS LONG) - CAST(chegada_prevista AS LONG)) / 60.0) AS INT)
       THEN 1 ELSE 0 END) AS atraso_chegada_inconsistente,
  SUM(CASE WHEN atraso_partida_min IS NOT NULL AND atraso_chegada_min IS NOT NULL
       AND minutos_recuperados IS NOT NULL
       AND minutos_recuperados != (atraso_partida_min - atraso_chegada_min)
       THEN 1 ELSE 0 END) AS minutos_recuperados_inconsistente
FROM voebem.silver.vra;

## 5. Regras de Negócio

Verificações de coerência do domínio VRA: voos cancelados não devem ter realização, voos realizados devem ter partida/chegada real, atrasos extremos indicam problemas, e chegada não pode ocorrer antes da partida.

In [0]:
%sql
-- Voos CANCELADO com partida_real preenchida (inconsistencia)
SELECT
  COUNT(*) AS cancelados_com_partida_real,
  SUM(CASE WHEN chegada_real IS NOT NULL THEN 1 ELSE 0 END) AS cancelados_com_chegada_real
FROM voebem.silver.vra
WHERE UPPER(TRIM(situacao_voo)) = 'CANCELADO'
  AND partida_real IS NOT NULL;

In [0]:
%sql
-- Voos REALIZADO sem partida_real ou chegada_real (inconsistencia)
SELECT
  SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END) AS realizados_sem_partida_real,
  SUM(CASE WHEN chegada_real IS NULL THEN 1 ELSE 0 END) AS realizados_sem_chegada_real
FROM voebem.silver.vra
WHERE UPPER(TRIM(situacao_voo)) = 'REALIZADO';

In [0]:
%sql
-- Atrasos extremos (outliers): atraso > 1440 min (24h) ou < -1440 min
SELECT
  situacao_voo,
  COUNT(*) AS qtd_outliers,
  MIN(atraso_partida_min) AS min_atraso_partida,
  MAX(atraso_partida_min) AS max_atraso_partida,
  MIN(atraso_chegada_min) AS min_atraso_chegada,
  MAX(atraso_chegada_min) AS max_atraso_chegada
FROM voebem.silver.vra
WHERE (atraso_partida_min > 1440 OR atraso_partida_min < -1440
    OR atraso_chegada_min > 1440 OR atraso_chegada_min < -1440)
GROUP BY situacao_voo;

In [0]:
%sql
-- Top 20 voos com atraso de partida mais extremo
SELECT icao_empresa, numero_voo, codigo_di, partida_prevista, partida_real,
  atraso_partida_min, atraso_chegada_min, situacao_voo
FROM voebem.silver.vra
WHERE atraso_partida_min IS NOT NULL
ORDER BY ABS(atraso_partida_min) DESC
LIMIT 20;

In [0]:
%sql
-- Voos com chegada antes de partida (inconsistencia temporal)
SELECT icao_empresa, numero_voo, codigo_di, partida_real, chegada_real,
  situacao_voo, icao_origem, icao_destino
FROM voebem.silver.vra
WHERE partida_real IS NOT NULL AND chegada_real IS NOT NULL
  AND chegada_real < partida_real
LIMIT 20;

In [0]:
%sql
-- Origem e destino iguais (pode indicar voo circular ou erro de dado)
SELECT
  COUNT(*) AS qtd_origem_igual_destino,
  COUNT(DISTINCT icao_empresa) AS empresas_distintas
FROM voebem.silver.vra
WHERE icao_origem = icao_destino
  AND icao_origem IS NOT NULL;

## 6. Integridade Referencial

Verificação de que as chaves `icao_empresa`, `icao_origem` e `icao_destino` existem nas tabelas de dimensão correspondentes.

In [0]:
%sql
-- icao_empresa que nao existem em silver.empresas
SELECT v.icao_empresa, COUNT(*) AS qtd_etapas
FROM voebem.silver.vra v
LEFT JOIN voebem.silver.empresas e ON v.icao_empresa = e.icao
WHERE e.icao IS NULL
  AND v.icao_empresa IS NOT NULL
  AND TRIM(v.icao_empresa) != ''
GROUP BY v.icao_empresa
ORDER BY qtd_etapas DESC
LIMIT 20;

In [0]:
%sql
-- icao_origem que nao existem em silver.aerodromos
SELECT v.icao_origem AS icao, COUNT(*) AS qtd_etapas, 'origem' AS tipo
FROM voebem.silver.vra v
LEFT JOIN voebem.silver.aerodromos a ON v.icao_origem = a.icao
WHERE a.icao IS NULL
  AND v.icao_origem IS NOT NULL
  AND TRIM(v.icao_origem) != ''
GROUP BY v.icao_origem
UNION ALL
SELECT v.icao_destino AS icao, COUNT(*) AS qtd_etapas, 'destino' AS tipo
FROM voebem.silver.vra v
LEFT JOIN voebem.silver.aerodromos a ON v.icao_destino = a.icao
WHERE a.icao IS NULL
  AND v.icao_destino IS NOT NULL
  AND TRIM(v.icao_destino) != ''
GROUP BY v.icao_destino
ORDER BY qtd_etapas DESC
LIMIT 40;

## 7. Resumo e Recomendações

Consolidação dos principais indicadores de qualidade e padronização.

In [0]:
%sql
-- Resumo consolidado dos checks de data quality
WITH stats AS (
  SELECT
    COUNT(*) AS total_registros,
    MIN(partida_prevista_data) AS data_minima,
    MAX(partida_prevista_data) AS data_maxima,
    SUM(CASE WHEN icao_empresa IS NULL OR TRIM(icao_empresa) = '' THEN 1 ELSE 0 END) AS icao_empresa_ausente,
    SUM(CASE WHEN numero_voo IS NULL OR TRIM(numero_voo) = '' THEN 1 ELSE 0 END) AS numero_voo_ausente,
    SUM(CASE WHEN codigo_di IS NULL OR TRIM(codigo_di) = '' THEN 1 ELSE 0 END) AS codigo_di_ausente,
    SUM(CASE WHEN codigo_tipo_linha IS NULL OR TRIM(codigo_tipo_linha) = '' THEN 1 ELSE 0 END) AS codigo_tipo_linha_ausente,
    SUM(CASE WHEN icao_origem IS NULL OR TRIM(icao_origem) = '' THEN 1 ELSE 0 END) AS icao_origem_ausente,
    SUM(CASE WHEN icao_destino IS NULL OR TRIM(icao_destino) = '' THEN 1 ELSE 0 END) AS icao_destino_ausente,
    SUM(CASE WHEN situacao_voo IS NULL OR TRIM(situacao_voo) = '' THEN 1 ELSE 0 END) AS situacao_voo_ausente,
    SUM(CASE WHEN UPPER(TRIM(situacao_voo)) = 'REALIZADO' THEN 1 ELSE 0 END) AS qtd_realizado,
    SUM(CASE WHEN UPPER(TRIM(situacao_voo)) = 'CANCELADO' THEN 1 ELSE 0 END) AS qtd_cancelado,
    SUM(CASE WHEN UPPER(TRIM(situacao_voo)) = 'REALIZADO' AND partida_real IS NULL THEN 1 ELSE 0 END) AS realizado_sem_partida,
    SUM(CASE WHEN UPPER(TRIM(situacao_voo)) = 'CANCELADO' AND partida_real IS NOT NULL THEN 1 ELSE 0 END) AS cancelado_com_partida,
    SUM(CASE WHEN partida_real IS NOT NULL AND chegada_real IS NOT NULL AND chegada_real < partida_real THEN 1 ELSE 0 END) AS chegada_antes_partida,
    SUM(CASE WHEN atraso_partida_min IS NOT NULL AND ABS(atraso_partida_min) > 1440 THEN 1 ELSE 0 END) AS atraso_extremo_partida,
    SUM(CASE WHEN icao_origem = icao_destino AND icao_origem IS NOT NULL THEN 1 ELSE 0 END) AS origem_igual_destino
  FROM voebem.silver.vra
)
SELECT
  total_registros,
  data_minima,
  data_maxima,
  qtd_realizado,
  qtd_cancelado,
  icao_empresa_ausente,
  numero_voo_ausente,
  codigo_di_ausente,
  codigo_tipo_linha_ausente,
  icao_origem_ausente,
  icao_destino_ausente,
  situacao_voo_ausente,
  realizado_sem_partida,
  cancelado_com_partida,
  chegada_antes_partida,
  atraso_extremo_partida,
  origem_igual_destino
FROM stats;